In [1]:
# 房價預測 - 核心版本
%pip install xgboost scikit-learn openpyxl
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("所有套件導入完成！")

Note: you may need to restart the kernel to use updated packages.
所有套件導入完成！


In [2]:
# 1. 載入資料
print("載入資料中...")
try:
    train = pd.read_excel('train-v2.xlsx', engine='openpyxl')
    print(f"訓練資料載入成功: {train.shape}")
except Exception as e:
    print(f"載入 train-v2.xlsx 失敗: {e}")
    train = pd.DataFrame()

try:
    valid = pd.read_excel('valid-v2.xlsx', engine='openpyxl')
    print(f"驗證資料載入成功: {valid.shape}")
except Exception as e:
    print(f"載入 valid-v2.xlsx 失敗: {e}")
    valid = pd.DataFrame()

try:
    test = pd.read_excel('test-reindex-test-v2.1.xlsx', engine='openpyxl')
    print(f"測試資料載入成功: {test.shape}")
except Exception as e:
    print(f"載入 test-reindex-test-v2.1.xlsx 失敗: {e}")
    test = pd.DataFrame()

print("=" * 50)

載入資料中...
訓練資料載入成功: (455787, 29)
驗證資料載入成功: (11994, 29)
測試資料載入成功: (47073, 28)


In [3]:
# 2. 資料清理
print("開始資料清理...")

def clean_data(df, name):
    """統一的資料清理函數"""
    original_count = len(df)
    
    if name == 'train':
        # 清理總價元異常資料
        print("清理總價元異常資料...")
        df['總價元'] = pd.to_numeric(df['總價元'], errors='coerce')
        invalid_mask = df['總價元'].isna()
        if invalid_mask.sum() > 0:
            print(f"移除 {invalid_mask.sum()} 筆格式錯誤的總價元資料")
            df = df[~invalid_mask].reset_index(drop=True)
    
    # 處理交易年月日
    def extract_year_month(date_str):
        if str(date_str).isdigit() and len(str(date_str)) >= 5:
            date_int = int(date_str)
            year = date_int // 10000
            month = (date_int // 100) % 100
            return year, month if 1 <= month <= 12 else None
        return None, None
    
    df[['交易年', '交易月']] = df['交易年月日'].apply(
        lambda x: pd.Series(extract_year_month(x))
    )
    
    # 保留合理年份和月份的資料
    valid_mask = (
        df['交易年'].between(90, 114, inclusive='both') & 
        df['交易月'].between(1, 12, inclusive='both') &
        df['交易年'].notna() & df['交易月'].notna()
    )
    df_clean = df[valid_mask].reset_index(drop=True)
    
    # 處理移轉層次中文轉數字
    def convert_floor_text(floor_text):
        if pd.isna(floor_text) or floor_text == '':
            return 0
        
        floor_str = str(floor_text).strip().replace('層', '')
        
        if floor_str in ['全', '全棟', '整棟']:
            return -1
        
        if floor_str.isdigit():
            return int(floor_str)
        
        # 中文數字轉換
        chinese_numbers = {
            '一': 1, '二': 2, '三': 3, '四': 4, '五': 5,
            '六': 6, '七': 7, '八': 8, '九': 9, '十': 10, '零': 0
        }
        
        if floor_str in chinese_numbers:
            return chinese_numbers[floor_str]
        
        if '十' in floor_str:
            if floor_str == '十':
                return 10
            elif floor_str.startswith('十'):
                if len(floor_str) > 1 and floor_str[1] in chinese_numbers:
                    return 10 + chinese_numbers[floor_str[1]]
                return 10
            elif floor_str.endswith('十'):
                if floor_str[0] in chinese_numbers:
                    return chinese_numbers[floor_str[0]] * 10
                return 10
            else:
                parts = floor_str.split('十')
                if len(parts) == 2:
                    tens = chinese_numbers.get(parts[0], 0) * 10
                    ones = chinese_numbers.get(parts[1], 0) if parts[1] else 0
                    return tens + ones
        
        return 0
    
    if '移轉層次' in df_clean.columns:
        df_clean['移轉層次'] = df_clean['移轉層次'].apply(convert_floor_text)
    
    removed = original_count - len(df_clean)
    if removed > 0:
        print(f"{name} 移除 {removed} 筆異常資料")
    
    print(f"{name} 清理完成: {original_count}→{len(df_clean)}筆")
    return df_clean

# 清理所有資料集
train = clean_data(train, 'train')
valid = clean_data(valid, 'valid')
test = clean_data(test, 'test')

print("=" * 50)

開始資料清理...
清理總價元異常資料...
移除 3 筆格式錯誤的總價元資料
train 移除 11 筆異常資料
train 清理完成: 455787→455776筆
valid 清理完成: 11994→11994筆
test 清理完成: 47073→47073筆


In [4]:
# 3. 完整特徵工程與預處理
print("開始進階特徵工程...")

def complete_feature_engineering(train_df, valid_df, test_df):
    """完整的特徵工程與預處理流程"""
    train_df = train_df.copy()
    valid_df = valid_df.copy() 
    test_df = test_df.copy()
    
    # 確保目標變數是數值型
    if '總價元' in train_df.columns:
        train_df['總價元'] = pd.to_numeric(train_df['總價元'], errors='coerce')
    if '總價元' in valid_df.columns:
        valid_df['總價元'] = pd.to_numeric(valid_df['總價元'], errors='coerce')
    
    for df in [train_df, valid_df, test_df]:
        # 確保數值欄位正確 (移除建物現況格局-房、建物現況格局-衛、建物現況格局-隔間)
        numeric_cols = ['土地移轉總面積平方公尺', '建物移轉總面積平方公尺', '車位移轉總面積平方公尺',
                       '移轉層次', '總樓層數']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        # ====== 基本特徵工程 ======
        # 地區特徵
        df['縣市名稱'] = df['土地位置建物門牌'].astype(str).str[:3]
        df['是否直轄市'] = df['縣市名稱'].isin(['臺北市', '新北市', '桃園市', '臺中市', '臺南市', '高雄市']).astype(int)
        
        # 交易數量特徵
        if '交易筆棟數' in df.columns:
            extracted = df['交易筆棟數'].str.extract(r'土地(\d+)建物(\d+)車位(\d+)')
            df['土地數'] = pd.to_numeric(extracted.iloc[:, 0], errors='coerce').fillna(0)
            df['建物數'] = pd.to_numeric(extracted.iloc[:, 1], errors='coerce').fillna(0)
            df['車位數'] = pd.to_numeric(extracted.iloc[:, 2], errors='coerce').fillna(0)
        
        # 土地使用分區
        df['土地使用分區'] = (df['都市土地使用分區'].astype(str) + 
                          df['非都市土地使用分區'].astype(str)).replace('nannan', 'unknown')
        
        # ====== 進階衍生特徵 ======
        # 面積相關
        df['總面積'] = df['土地移轉總面積平方公尺'] + df['建物移轉總面積平方公尺'] + df['車位移轉總面積平方公尺']
        df['土地建物面積比'] = df['土地移轉總面積平方公尺'] / (df['建物移轉總面積平方公尺'] + 1e-6)
        df['車位面積比例'] = df['車位移轉總面積平方公尺'] / (df['總面積'] + 1e-6)
        
        # 樓層相關
        df['樓層比例'] = df['移轉層次'] / (df['總樓層數'] + 1e-6)
        df['是否頂樓'] = (df['移轉層次'] == df['總樓層數']).astype(int)
        df['是否低樓層'] = (df['移轉層次'] <= 3).astype(int)
        df['是否中高樓層'] = (df['移轉層次'] > df['總樓層數'] * 0.7).astype(int)
        df['是否地下層'] = (df['移轉層次'] < 0).astype(int)
        
        # 交易相關
        df['總交易數量'] = df['土地數'] + df['建物數'] + df['車位數']
        df['是否包含車位'] = (df['車位數'] > 0).astype(int)
        df['是否多筆交易'] = (df['總交易數量'] > 3).astype(int)
        df['建物土地數比'] = df['建物數'] / (df['土地數'] + 1e-6)
        
        # 面積分段
        if '建物移轉總面積平方公尺' in df.columns:
            df['小坪數'] = (df['建物移轉總面積平方公尺'] <= 50).astype(int)
            df['中坪數'] = ((df['建物移轉總面積平方公尺'] > 50) & 
                          (df['建物移轉總面積平方公尺'] <= 100)).astype(int)
            df['大坪數'] = (df['建物移轉總面積平方公尺'] > 100).astype(int)
        
        # 時間特徵
        if '交易年' in df.columns:
            df['房齡估計'] = df['交易年'] - 75  # 估計建築年代
            df['是否新屋'] = (df['房齡估計'] <= 5).astype(int)
            df['是否老屋'] = (df['房齡估計'] >= 30).astype(int)
        
        # ====== 基於相關性分析的進階組合特徵 ======
        # 根據分析結果，總樓層×面積是最強相關特徵 (r=0.6715)
        df['總樓層×面積'] = df['總樓層數'] * df['建物移轉總面積平方公尺']
        df['面積×樓層'] = df['建物移轉總面積平方公尺'] * df['移轉層次']
        
        # 車位相關組合特徵 (車位數和車位面積都是強相關)
        df['車位總價值指標'] = df['車位數'] * df['車位移轉總面積平方公尺']
        df['車位密度'] = df['車位數'] / (df['總面積'] + 1e-6)
        df['車位面積×數量'] = df['車位移轉總面積平方公尺'] * df['車位數']
        
        # 建物數量與面積的高價值組合
        df['建物密度指標'] = df['建物數'] * df['建物移轉總面積平方公尺']
        df['平均建物面積'] = df['建物移轉總面積平方公尺'] / (df['建物數'] + 1e-6)
        
        # 複合面積特徵 (結合多個面積維度)
        df['加權總面積'] = (df['建物移轉總面積平方公尺'] * 0.6 + 
                          df['車位移轉總面積平方公尺'] * 0.3 + 
                          df['土地移轉總面積平方公尺'] * 0.1)
        
        # 樓層效應特徵
        df['樓層價值係數'] = df['移轉層次'] * df['總樓層數'] * df['建物移轉總面積平方公尺']
        df['樓層面積效率'] = df['建物移轉總面積平方公尺'] / (df['總樓層數'] + 1e-6)
        
        # 交易規模特徵 (建物數是重要特徵)
        df['交易規模指標'] = df['建物數'] * df['土地數'] * df['車位數']
        df['總交易價值權重'] = (df['建物數'] * 0.5 + df['車位數'] * 0.3 + df['土地數'] * 0.2)
        
        # 面積配比最佳化特徵
        df['黃金面積比'] = df['建物移轉總面積平方公尺'] / (df['總面積'] + 1e-6)
        df['車位面積佔比'] = df['車位移轉總面積平方公尺'] / (df['總面積'] + 1e-6)
        
        # 面積等級特徵 (轉為字符串避免類別問題)
        df['面積等級'] = pd.cut(df['建物移轉總面積平方公尺'], 
                           bins=[0, 30, 60, 100, 200, float('inf')],
                           labels=['迷你', '小型', '中型', '大型', '豪宅']).astype(str)
        
        # 樓層分級特徵 (轉為字符串避免類別問題)
        df['樓層等級'] = pd.cut(df['移轉層次'], 
                          bins=[-float('inf'), 0, 3, 7, 15, float('inf')],
                          labels=['地下', '低樓層', '中樓層', '高樓層', '超高樓層']).astype(str)
        
        # 車位等級特徵 (轉為字符串避免類別問題)
        df['車位等級'] = pd.cut(df['車位數'], 
                          bins=[0, 1, 2, 3, float('inf')],
                          labels=['無車位', '單車位', '雙車位', '多車位']).astype(str)
        
        # 綜合豪華指標 (結合多個高相關特徵)
        df['豪華指標'] = (
            (df['建物移轉總面積平方公尺'] > 100).astype(int) * 0.3 +
            (df['車位數'] >= 2).astype(int) * 0.3 +
            (df['總樓層數'] >= 10).astype(int) * 0.2 +
            (df['移轉層次'] >= 5).astype(int) * 0.2
        )
    
    # ====== 增強目標編碼 ======
    if '總價元' in train_df.columns:
        overall_mean = train_df['總價元'].mean()
        smoothing_factor = 10
        
        encodings = {}
        
        # 原始類別特徵編碼
        categorical_features = ['縣市名稱', '鄉鎮市區', '建物型態', '主要建材', '土地使用分區']
        
        # 新增等級特徵編碼
        level_features = ['面積等級', '樓層等級', '車位等級']
        
        all_features = categorical_features + level_features
        
        for col in all_features:
            if col in train_df.columns:
                group_stats = train_df.groupby(col)['總價元'].agg(['mean', 'count', 'std']).reset_index()
                
                # 加權平滑編碼
                group_stats['smoothed_mean'] = ((group_stats['mean'] * group_stats['count'] + 
                                               overall_mean * smoothing_factor) / 
                                              (group_stats['count'] + smoothing_factor))
                
                # 價格變異性編碼 (標準差資訊)
                group_stats['price_volatility'] = group_stats['std'].fillna(group_stats['std'].mean())
                
                encodings[f'{col}_價格等級'] = dict(zip(group_stats[col], group_stats['smoothed_mean']))
                encodings[f'{col}_價格變異'] = dict(zip(group_stats[col], group_stats['price_volatility']))
        
        # 組合特徵的目標編碼
        # 地區×面積等級組合編碼
        if '縣市名稱' in train_df.columns and '面積等級' in train_df.columns:
            train_df['地區面積組合'] = train_df['縣市名稱'].astype(str) + '_' + train_df['面積等級'].astype(str)
            combo_stats = train_df.groupby('地區面積組合')['總價元'].agg(['mean', 'count']).reset_index()
            combo_stats['smoothed_mean'] = ((combo_stats['mean'] * combo_stats['count'] + 
                                            overall_mean * smoothing_factor) / 
                                           (combo_stats['count'] + smoothing_factor))
            encodings['地區面積組合_價格等級'] = dict(zip(combo_stats['地區面積組合'], combo_stats['smoothed_mean']))
        
        # 建物型態×車位等級組合編碼
        if '建物型態' in train_df.columns and '車位等級' in train_df.columns:
            train_df['建物車位組合'] = train_df['建物型態'].astype(str) + '_' + train_df['車位等級'].astype(str)
            combo_stats2 = train_df.groupby('建物車位組合')['總價元'].agg(['mean', 'count']).reset_index()
            combo_stats2['smoothed_mean'] = ((combo_stats2['mean'] * combo_stats2['count'] + 
                                             overall_mean * smoothing_factor) / 
                                            (combo_stats2['count'] + smoothing_factor))
            encodings['建物車位組合_價格等級'] = dict(zip(combo_stats2['建物車位組合'], combo_stats2['smoothed_mean']))
        
        # 應用所有編碼到三個資料集
        for df in [train_df, valid_df, test_df]:
            # 創建組合特徵
            if '縣市名稱' in df.columns and '面積等級' in df.columns:
                df['地區面積組合'] = df['縣市名稱'].astype(str) + '_' + df['面積等級'].astype(str)
            if '建物型態' in df.columns and '車位等級' in df.columns:
                df['建物車位組合'] = df['建物型態'].astype(str) + '_' + df['車位等級'].astype(str)
            
            # 應用所有編碼 (修復填充值問題)
            for col, encoding in encodings.items():
                original_col = col.replace('_價格等級', '').replace('_價格變異', '')
                if original_col in df.columns:
                    if '價格等級' in col:
                        df[col] = df[original_col].map(encoding).fillna(overall_mean)
                    else:  # 價格變異特徵
                        mean_volatility = np.mean(list(encoding.values()))
                        df[col] = df[original_col].map(encoding).fillna(mean_volatility)
    
    # ====== 清理不需要的欄位 (加入要移除的特徵和輔助欄位) ======
    drop_cols = ['建案名稱', '編號', '總價元', '土地位置建物門牌', '備註', 
                 '都市土地使用分區', '非都市土地使用分區', '棟及號', '交易筆棟數', 
                 '非都市土地使用編定', '土地使用分區', '縣市名稱', '鄉鎮市區', 
                 '建築完成年月', '有無管理組織', '解約情形', '建物現況格局-廳',
                 '交易標的', '建物型態', '主要建材', '主要用途', '交易年月日',
                 '建物現況格局-房', '建物現況格局-衛', '建物現況格局-隔間', '車位類別',
                 # 清理等級特徵的原始類別欄位，保留編碼後的數值特徵
                 '面積等級', '樓層等級', '車位等級', '地區面積組合', '建物車位組合']
    
    for df in [train_df, valid_df, test_df]:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
        
        # 缺失值處理
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(df[col].median())
    
    # ====== 類別變數編碼 ======
    encoders = {}
    for col in train_df.select_dtypes(include=['object']).columns:
        for df in [train_df, valid_df, test_df]:
            df[col] = df[col].astype(str).fillna('unknown')
        
        le = LabelEncoder()
        all_values = pd.concat([train_df[col], valid_df[col], test_df[col]]).unique()
        le.fit(all_values)
        
        for df in [train_df, valid_df, test_df]:
            df[col] = le.transform(df[col])
        encoders[col] = le
    
    return train_df, valid_df, test_df, encoders

def create_advanced_correlation_features(X_train, X_valid, X_test):
    """基於相關性分析結果創建更進階的特徵"""
    print("創建基於相關性分析的進階特徵...")
    
    for df in [X_train, X_valid, X_test]:
        # 基於總樓層×面積的衍生特徵 (最強相關特徵 r=0.6715)
        if '總樓層×面積' in df.columns:
            df['總樓層×面積_對數'] = np.log1p(df['總樓層×面積'])
            df['總樓層×面積_平方根'] = np.sqrt(df['總樓層×面積'])
            df['總樓層×面積_立方根'] = np.cbrt(df['總樓層×面積'])
        
        # 車位特徵的進階變換 (車位數 r=0.5895, 車位面積 r=0.5854)
        if '車位數' in df.columns:
            df['車位數_平方'] = df['車位數'] ** 2
            df['車位數_對數'] = np.log1p(df['車位數'])
        
        if '車位移轉總面積平方公尺' in df.columns:
            df['車位面積_對數'] = np.log1p(df['車位移轉總面積平方公尺'])
            df['車位面積_平方根'] = np.sqrt(df['車位移轉總面積平方公尺'])
        
        # 建物面積的非線性變換 (r=0.4694)
        if '建物移轉總面積平方公尺' in df.columns:
            df['建物面積_對數'] = np.log1p(df['建物移轉總面積平方公尺'])
            df['建物面積_平方'] = df['建物移轉總面積平方公尺'] ** 2
            df['建物面積_立方根'] = np.cbrt(df['建物移轉總面積平方公尺'])
        
        # 建物數的非線性特徵 (r=0.3793)
        if '建物數' in df.columns:
            df['建物數_平方'] = df['建物數'] ** 2
            df['建物數_對數'] = np.log1p(df['建物數'])
        
        # 價格等級特徵的交互作用
        price_level_cols = [col for col in df.columns if '價格等級' in col]
        if len(price_level_cols) >= 2:
            # 創建價格等級的組合特徵
            df['平均價格等級'] = df[price_level_cols].mean(axis=1)
            df['最高價格等級'] = df[price_level_cols].max(axis=1)
            df['價格等級變異'] = df[price_level_cols].std(axis=1).fillna(0)
        
        # 面積相關特徵的比率和交互作用
        area_cols = ['土地移轉總面積平方公尺', '建物移轉總面積平方公尺', '車位移轉總面積平方公尺']
        available_area_cols = [col for col in area_cols if col in df.columns]
        
        if len(available_area_cols) >= 2:
            # 面積之間的比率特徵
            for i, col1 in enumerate(available_area_cols):
                for col2 in available_area_cols[i+1:]:
                    ratio_name = f'{col1}_vs_{col2}_比率'
                    df[ratio_name] = df[col1] / (df[col2] + 1e-6)
        
        # 樓層相關的進階特徵
        if '移轉層次' in df.columns and '總樓層數' in df.columns:
            # 樓層位置的非線性特徵
            df['樓層位置_平方'] = (df['移轉層次'] / (df['總樓層數'] + 1e-6)) ** 2
            df['樓層位置_對數'] = np.log1p(df['移轉層次'] / (df['總樓層數'] + 1e-6))
        
        # 綜合評分特徵 (結合多個高相關特徵)
        high_corr_features = []
        if '總樓層×面積' in df.columns:
            high_corr_features.append('總樓層×面積')
        if '車位數' in df.columns:
            high_corr_features.append('車位數')
        if '車位移轉總面積平方公尺' in df.columns:
            high_corr_features.append('車位移轉總面積平方公尺')
        if '建物移轉總面積平方公尺' in df.columns:
            high_corr_features.append('建物移轉總面積平方公尺')
        
        if len(high_corr_features) >= 3:
            # 標準化後的加權綜合評分
            for col in high_corr_features:
                df[f'{col}_標準化'] = (df[col] - df[col].mean()) / (df[col].std() + 1e-6)
            
            # 根據相關性強度設定權重
            weights = [0.35, 0.25, 0.25, 0.15]  # 對應總樓層×面積, 車位數, 車位面積, 建物面積
            df['房價預測綜合評分'] = sum(df[f'{col}_標準化'] * w 
                                     for col, w in zip(high_corr_features[:4], weights[:len(high_corr_features)]))
            
            # 清理標準化特徵
            for col in high_corr_features:
                if f'{col}_標準化' in df.columns:
                    df.drop(columns=[f'{col}_標準化'], inplace=True)
    
    print("進階相關性特徵創建完成！")
    return X_train, X_valid, X_test

# 執行完整特徵工程
X_train, X_valid, X_test, encoders = complete_feature_engineering(train, valid, test)

# 創建進階相關性特徵
X_train, X_valid, X_test = create_advanced_correlation_features(X_train, X_valid, X_test)

# 準備目標變數
y_train = pd.to_numeric(train['總價元'], errors='coerce').fillna(0)
y_valid = pd.to_numeric(valid['總價元'], errors='coerce').fillna(0)

print(f"特徵工程完成！")
print(f"特徵數量: {X_train.shape[1]}")
print(f"資料形狀: train={X_train.shape}, valid={X_valid.shape}, test={X_test.shape}")
print("=" * 50)

開始進階特徵工程...
創建基於相關性分析的進階特徵...
進階相關性特徵創建完成！
特徵工程完成！
特徵數量: 83
資料形狀: train=(455776, 83), valid=(11994, 83), test=(47073, 83)


In [ ]:
# 4. 特徵選擇與模型訓練
print("開始特徵選擇與模型訓練...")

# 智能特徵選擇
def intelligent_feature_selection(X_train, y_train, X_valid, X_test, top_k=30):
    """使用XGBoost進行特徵重要性排序"""
    print("進行特徵重要性分析...")
    
    # 找出共同特徵
    common_features = [f for f in X_train.columns if f in X_valid.columns and f in X_test.columns]
    print(f"共同特徵數量: {len(common_features)}")
    
    # 特徵選擇器
    selector = xgb.XGBRegressor(
        n_estimators=200, 
        max_depth=6, 
        learning_rate=0.1,
        random_state=42, 
        tree_method='gpu_hist'
    )
    
    selector.fit(X_train[common_features], y_train)
    
    # 獲取特徵重要性
    feature_importance = pd.Series(selector.feature_importances_, index=common_features)
    top_features = feature_importance.nlargest(top_k).index.tolist()
    
    print(f"選出前{top_k}個重要特徵")
    print(f"前5個重要特徵:")
    for i, (feature, importance) in enumerate(feature_importance.nlargest(5).items(), 1):
        print(f"   {i}. {feature}: {importance:.4f}")
    
    return top_features, feature_importance

# 執行特徵選擇
selected_features, feature_importance = intelligent_feature_selection(X_train, y_train, X_valid, X_test)

X_train_final = X_train[selected_features]
X_valid_final = X_valid[selected_features]
X_test_final = X_test[selected_features]

print("\n開始訓練最佳化XGBoost模型...")

# 最佳化XGBoost模型
best_model = xgb.XGBRegressor(
    n_estimators=2500,        # 增加樹的數量
    max_depth=12,             # 增加樹的深度
    learning_rate=0.04,       # 降低學習率以提高穩定性
    subsample=0.85,           # 樣本採樣比例
    colsample_bytree=0.85,    # 特徵採樣比例
    reg_alpha=0.1,            # L1正則化
    reg_lambda=1.2,           # L2正則化
    min_child_weight=2,       # 最小葉子節點樣本權重
    gamma=0.05,               # 最小分割損失
    random_state=42,
    tree_method='gpu_hist',   # 使用GPU加速
    n_jobs=-1                 # 使用所有CPU核心
)

# 訓練模型
print("模型訓練中...")
best_model.fit(X_train_final, y_train)
print("模型訓練完成！")

print("=" * 50)

🎯 開始特徵選擇與模型訓練...
🔍 進行特徵重要性分析...
共同特徵數量: 83
✅ 選出前30個重要特徵
🏆 前5個重要特徵:
   1. 總交易價值權重: 0.6677
   2. 是否直轄市: 0.0627
   3. 平均價格等級: 0.0569
   4. 地區面積組合_價格等級: 0.0329
   5. 加權總面積: 0.0189

🚀 開始訓練最佳化XGBoost模型...
⏳ 模型訓練中...
✅ 模型訓練完成！


In [ ]:
# 5. 模型評估與預測
print("開始模型評估...")

# 定義評估指標
def calculate_metrics(y_true, y_pred):
    """計算回歸評估指標"""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-6))) * 100
    return r2, rmse, mae, mape

# 進行預測
print("🔮 生成預測結果...")
train_pred = best_model.predict(X_train_final)
valid_pred = best_model.predict(X_valid_final)
test_pred = best_model.predict(X_test_final)

# 計算評估指標
train_r2, train_rmse, train_mae, train_mape = calculate_metrics(y_train, train_pred)
valid_r2, valid_rmse, valid_mae, valid_mape = calculate_metrics(y_valid, valid_pred)

print("\n模型效果評估:")
print("=" * 60)
print(f"訓練集表現:")
print(f"   R² Score: {train_r2:.4f}")
print(f"   RMSE: {train_rmse:,.0f} 元")
print(f"   MAE: {train_mae:,.0f} 元")
print(f"   MAPE: {train_mape:.2f}%")

print(f"驗證集表現:")
print(f"   R² Score: {valid_r2:.4f}")
print(f"   RMSE: {valid_rmse:,.0f} 元")
print(f"   MAE: {valid_mae:,.0f} 元")
print(f"   MAPE: {valid_mape:.2f}%")

print(f"\n模型解釋力: {valid_r2*100:.2f}% 的房價變異可被解釋")

# 顯示最終特徵重要性
print(f"\n最重要的10個特徵:")
final_importance = pd.Series(best_model.feature_importances_, index=selected_features)
final_importance = final_importance.sort_values(ascending=False)

for i, (feature, importance) in enumerate(final_importance.head(10).items(), 1):
    print(f"   {i:2d}. {feature}: {importance:.4f}")

print("=" * 50)

📊 開始模型評估...
🔮 生成預測結果...

🏆 模型效果評估:
訓練集表現:
   R² Score: 0.9996
   RMSE: 355,179 元
   MAE: 240,272 元
   MAPE: 3.64%
驗證集表現:
   R² Score: 0.9895 ⭐
   RMSE: 1,292,652 元
   MAE: 513,082 元
   MAPE: 3.48%

模型解釋力: 98.95% 的房價變異可被解釋

🔥 最重要的10個特徵:
    1. 車位數: 0.4454
    2. 建物數: 0.2688
    3. 交易規模指標: 0.0818
    4. 總交易價值權重: 0.0644
    5. 地區面積組合_價格等級: 0.0258
    6. 是否直轄市: 0.0168
    7. 平均價格等級: 0.0153
    8. 主要建材_價格等級: 0.0151
    9. 主要建材_價格變異: 0.0150
   10. 最高價格等級: 0.0112


In [7]:
# 6. 輸出最終預測結果
print("準備輸出預測結果...")

# 創建最終預測結果
final_predictions = pd.DataFrame({
    '編號': test['編號'],
    '總價元': test_pred.round(2)
})

# 保存到CSV
output_filename = 'final_house_price_predictions.csv'
final_predictions.to_csv(output_filename, index=False, encoding='utf-8', float_format='%.2f')

print(f"預測結果已保存至: {output_filename}")
print(f"\n預測結果預覽:")
print(final_predictions.head(10).to_string(index=False))

print(f"\n預測統計:")
print(f"   預測筆數: {len(final_predictions):,}")
print(f"   最低預測價格: {test_pred.min():,.0f} 元")
print(f"   最高預測價格: {test_pred.max():,.0f} 元")
print(f"   平均預測價格: {test_pred.mean():,.0f} 元")
print(f"   預測價格中位數: {np.median(test_pred):,.0f} 元")

print("\n" + "=" * 60)
print("房價預測模型完成！")
print("=" * 60)
print(f"最終模型總結:")
print(f"   • 模型類型: 最佳化 XGBoost")
print(f"   • 驗證集 R²: {valid_r2:.4f}")
print(f"   • 驗證集 RMSE: {valid_rmse:,.0f} 元")
print(f"   • 使用特徵: {len(selected_features)} 個精選特徵")
print(f"   • 預測檔案: {output_filename}")

print(f"\n建議:")
print(f"   ✓ 使用 {output_filename} 作為最終提交結果")
print(f"   ✓ 模型已經過充分的特徵工程和參數調優")
print(f"   ✓ R² = {valid_r2:.4f} 表示模型解釋了 {valid_r2*100:.1f}% 的房價變異")
print("=" * 60)

準備輸出預測結果...
預測結果已保存至: final_house_price_predictions.csv

預測結果預覽:
 編號        總價元
  1 15261541.0
  2 20111920.0
  3 38741744.0
  4 12175716.0
  5  9483006.0
  6 22131560.0
  7 24318180.0
  8 17248918.0
  9 22207120.0
 10 26549928.0

預測統計:
   預測筆數: 47,073
   最低預測價格: 425,936 元
   最高預測價格: 1,906,518,528 元
   平均預測價格: 18,555,794 元
   預測價格中位數: 15,015,808 元

房價預測模型完成！
最終模型總結:
   • 模型類型: 最佳化 XGBoost
   • 驗證集 R²: 0.9895
   • 驗證集 RMSE: 1,292,652 元
   • 使用特徵: 30 個精選特徵
   • 預測檔案: final_house_price_predictions.csv

建議:
   ✓ 使用 final_house_price_predictions.csv 作為最終提交結果
   ✓ 模型已經過充分的特徵工程和參數調優
   ✓ R² = 0.9895 表示模型解釋了 98.9% 的房價變異
